## IMPORT LIBRAIRIES

In [1]:
import os
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

### Super function

In [ ]:
def add_noise_to_dataset(df, columns=None, noise_fraction=0.12, possible_values_dict=None):
    """
    Ajoute du bruit à plusieurs colonnes d'un DataFrame d'un seul coup.

    :param df: Le DataFrame de test original.
    :param columns: Liste des colonnes à bruiter. Si None, applique à toutes les colonnes.
    :param noise_fraction: La proportion de données à modifier par colonne (ex: 0.05).
    :param possible_values_dict: (Optionnel) Un dictionnaire {nom_colonne: [valeurs_possibles]}.
    :return: Un nouveau DataFrame avec le bruit ajouté.
    """
    df_noisy = df.copy()

    # Si aucune colonne n'est spécifiée, on prend toutes les colonnes du dataset
    if columns is None:
        columns = df_noisy.columns

    n_rows = len(df_noisy)
    n_noise = int(n_rows * noise_fraction)

    for col in columns:
        # 1. Déterminer les valeurs possibles pour cette colonne spécifique
        if possible_values_dict and col in possible_values_dict:
            possible_values = possible_values_dict[col]
        else:
            # Détection automatique : on prend les valeurs uniques existantes dans la colonne
            possible_values = df_noisy[col].dropna().unique()

        # Si la colonne est vide ou n'a qu'une seule valeur possible, on l'ignore
        if len(possible_values) <= 1:
            continue

        # 2. Sélectionner les lignes à modifier (indices au hasard)
        noise_indices = np.random.choice(df_noisy.index, size=n_noise, replace=False)

        # 3. Générer les nouvelles valeurs aléatoires parmi les choix possibles pour cette colonne
        random_new_values = np.random.choice(possible_values, size=n_noise)

        # 4. Appliquer le bruit
        df_noisy.loc[noise_indices, col] = random_new_values

    return df_noisy

## IMPORT DATASETS

In [4]:
df_primary = pd.read_csv("../data/table_dataset/primary_data.csv", sep=";")
df_secondary = pd.read_csv("../data/table_dataset/secondary_data.csv", sep=";")
df_mushnames = pd.read_csv("../data/table_dataset/species_names.csv", sep =";")

### Cleaning data tabular 

#### Data for edible classification

In [6]:
# Species name from primary, add to secondary
primary_name_df = df_primary[['family','name']]
df_primname_rep = primary_name_df.loc[primary_name_df.index.repeat(353)].reset_index(drop=True)

# Clean names
data_secondary_labelled = pd.concat([df_secondary, df_primname_rep], axis=1)
data_secondary_labelled['family'] = data_secondary_labelled['family'].str.replace(" Family", "", regex=False)
#
data_secondary_labelled['Common Name'] = data_secondary_labelled["family"] + " " + data_secondary_labelled["name"]
data_secondary_labelled

# Merge to scientific names
data_merge_scname = data_secondary_labelled.merge(df_mushnames, how='left', on='Common Name')
data_tabular_final = data_merge_scname.drop(columns=['family','name','Common Name'])
data_tabular_final.columns = data_tabular_final.columns.str.replace('-', '_').str.replace(' ', '_').str.lower()
data_tabular_final["class"] = (data_tabular_final["class"] == "p").astype(int)
data_tabular_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 61069 entries, 0 to 61068
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   class                 61069 non-null  int64  
 1   cap_diameter          61069 non-null  float64
 2   cap_shape             61069 non-null  str    
 3   cap_surface           46949 non-null  str    
 4   cap_color             61069 non-null  str    
 5   does_bruise_or_bleed  61069 non-null  str    
 6   gill_attachment       51185 non-null  str    
 7   gill_spacing          36006 non-null  str    
 8   gill_color            61069 non-null  str    
 9   stem_height           61069 non-null  float64
 10  stem_width            61069 non-null  float64
 11  stem_root             9531 non-null   str    
 12  stem_surface          22945 non-null  str    
 13  stem_color            61069 non-null  str    
 14  veil_type             3177 non-null   str    
 15  veil_color            7413 non

In [8]:
data_tabular_final.sample(frac=1, random_state=3).reset_index(drop=True)

,class,cap_diameter,cap_shape,cap_surface,cap_color,does_bruise_or_bleed,gill_attachment,gill_spacing,gill_color,stem_height,...,stem_surface,stem_color,veil_type,veil_color,has_ring,ring_type,spore_print_color,habitat,season,scientific_name
0,0,4.19,x,NaN,y,f,a,c,o,4.11,...,NaN,w,NaN,NaN,f,f,NaN,d,u,Russula claroflava
1,1,12.26,x,t,n,f,a,d,w,12.18,...,NaN,w,NaN,NaN,f,f,NaN,d,u,Russula foetens
2,1,2.58,b,NaN,k,f,a,NaN,n,8.69,...,NaN,g,NaN,w,f,f,k,g,a,Panaeolus papilionaceus
3,0,6.98,x,h,y,t,x,c,y,8.57,...,NaN,w,NaN,NaN,f,f,NaN,d,u,Russula lutea
4,0,8.21,f,s,k,f,a,c,w,4.76,...,NaN,w,NaN,NaN,f,f,NaN,d,a,Russula atropurpurea
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61064,1,7.45,x,i,n,f,a,c,n,7.33,...,i,n,NaN,k,t,z,k,d,u,Lacrymaria lacrymabunda
61065,1,5.77,f,k,y,f,p,NaN,n,3.52,...,k,n,NaN,NaN,t,f,NaN,d,u,Pseudoclitocybe cyathiformis
61066,0,1.58,c,g,n,f,a,NaN,g,4.85,...,NaN,g,NaN,NaN,f,f,NaN,l,a,Mycena galopus
61067,1,8.33,x,h,r,f,NaN,c,w,11.46,...,NaN,w,u,w,t,g,NaN,d,a,Amanita phalloides


#### Data for species classification

In [6]:
## Liste champignons (via noms scientifiques) dans les images
path_edible = "data/image_dataset/edible"
path_poisonous = "data/image_dataset/poisonous"

# Liste des sous-dossiers
list_edible = [f for f in os.listdir(path_edible)
                 if os.path.isdir(os.path.join(path_edible, f))]
list_poisonous = [f for f in os.listdir(path_poisonous)
                 if os.path.isdir(os.path.join(path_poisonous, f))]

# Création du DataFrame
df1 = pd.DataFrame(list_edible, columns=["scientific_name"])
df1['type'] = "edible"
df2 = pd.DataFrame(list_poisonous, columns=["scientific_name"])
df2['type'] = "poisonous"
df_concat = pd.concat([df1, df2], ignore_index=True)
df_concat['scientific_name'] = df_concat['scientific_name'].str.replace("_", " ", regex=False).str.replace("-", " ", regex=False)

# garder que le tabulaire dont l'espèce est présente dans les données d'image
data_tabular_image = data_tabular_final.merge(df_concat, how='inner', on='scientific_name')
data_tabular_image

,class,cap_diameter,cap_shape,cap_surface,cap_color,does_bruise_or_bleed,gill_attachment,gill_spacing,gill_color,stem_height,...,stem_color,veil_type,veil_color,has_ring,ring_type,spore_print_color,habitat,season,scientific_name,type
0,1,6.87,x,g,n,f,e,NaN,w,6.88,...,w,u,w,t,p,NaN,d,a,Amanita pantherina,poisonous
1,1,8.59,p,g,n,f,e,NaN,w,9.15,...,w,u,w,t,p,NaN,d,a,Amanita pantherina,poisonous
2,1,5.95,p,g,n,f,e,NaN,w,7.54,...,w,u,w,t,p,NaN,d,u,Amanita pantherina,poisonous
3,1,6.51,x,g,n,f,e,NaN,w,6.80,...,w,u,w,t,p,NaN,d,a,Amanita pantherina,poisonous
4,1,7.66,x,g,n,f,e,NaN,w,8.55,...,w,u,w,t,p,NaN,d,a,Amanita pantherina,poisonous
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16939,1,11.76,o,NaN,e,f,f,f,f,4.90,...,n,NaN,NaN,f,f,NaN,d,u,Sarcosphaera coronaria,poisonous
16940,1,8.54,o,NaN,e,f,f,f,f,3.72,...,n,NaN,NaN,f,f,NaN,d,u,Sarcosphaera coronaria,poisonous
16941,1,7.30,o,NaN,e,f,f,f,f,2.79,...,n,NaN,NaN,f,f,NaN,d,s,Sarcosphaera coronaria,poisonous
16942,1,10.40,o,NaN,n,f,f,f,f,3.82,...,n,NaN,NaN,f,f,NaN,d,u,Sarcosphaera coronaria,poisonous


### Train Test Split both datas

In [ ]:
# EDIBLE CLASSIFICATION
# prepare X and y

X = data_tabular_final.drop(columns=['class','gill_spacing','stem_root','stem_surface',
                                     'veil_type','veil_color','spore_print_color','scientific_name'])
y = data_tabular_final['class']

# TTS

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=3)


In [8]:
# SPECIES CLASSIFICATION
# prepare X and y

X_tabimage = data_tabular_image.drop(columns=['class','gill_spacing','stem_root','stem_surface',
                                     'veil_type','veil_color','spore_print_color','scientific_name'])
y_tabimage = data_tabular_image['scientific_name']

# TTS

X_tabimage_train, X_tabimage_test, y_tabimage_train, y_tabimage_test = train_test_split(X_tabimage,
                                                                            y_tabimage,
                                                                            test_size=0.3,
                                                                            random_state=3)

### Preprocess pipeline

In [ ]:
# pipeline num and cat
num_transformer = make_pipeline(SimpleImputer(strategy="median"), MinMaxScaler())
cat_transformer = make_pipeline(SimpleImputer(strategy='constant', fill_value='u'),
                                OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False))

# preprocess all
preproc_basic = make_column_transformer(
    (num_transformer, make_column_selector(dtype_include=np.number)),
    (cat_transformer, make_column_selector(dtype_exclude=np.number)),
    remainder='drop'
).set_output(transform="pandas")


(42748, 90)
(18321, 90)


## XGBOOST BASELINE - Edibility classification

In [ ]:
model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42)

In [17]:
pipe_baseline = make_pipeline(preproc_basic, model)
pipe_baseline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('xgbclassifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('pipeline-1', ...), ('pipeline-2', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of th

In [39]:
model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42)

pipe_baseline = make_pipeline(preproc_basic, model)

score_baseline = cross_val_score(pipe_baseline, X_train, y_train, cv=5, scoring='recall').mean()
score_baseline

np.float64(0.9992393384054227)

In [ ]:
pipe_baseline.fit(X_train, y_train)

importances = pipe_baseline[-1].feature_importances_
features = pipe_baseline[0].get_feature_names_out()

feat_imp = pd.Series(importances, index=features).sort_values(ascending=False)


In [60]:
dfimp = pd.DataFrame({
    "importances": importances,
    "features": features
})
dfimp["features"] = dfimp["features"].str.replace(r".*__", "", regex=True).str[:-2]

dfimp.groupby("features", as_index=False)["importances"]\
      .sum()\
      .sort_values(by="importances", ascending=False)

,features,importances
3,cap_surface,0.174634
11,stem_color,0.134027
5,gill_attachment,0.131807
0,cap_color,0.120689
6,gill_color,0.111649
2,cap_shape,0.105256
9,ring_type,0.095913
10,season,0.031434
7,habitat,0.027388
4,does_bruise_or_bleed,0.016865


In [38]:
#pipe_baseline.fit(X_train, y_train)
#pipe_baseline.predict(X_test)

## RandomForest BASELINE - Edibility classification

In [27]:
model = RandomForestClassifier()
pipe_rf = make_pipeline(preproc_basic, model)
score_rf = cross_val_score(pipe_rf, X_train, y_train, cv=5, scoring='accuracy').mean()
score_rf

np.float64(0.999836254573737)